In [7]:
# JupyterDash is deprecated and does not work with the newer Dash versions.
# Newer Dash can run inside Jupyter by itself, so we use Dash instead.
from dash import Dash

# Configure the necessary Python module imports for dashboard components
import dash_leaflet as dl
from dash import dcc
from dash import html
import plotly.express as px
from dash import dash_table
from dash.dependencies import Input, Output, State
import base64

# Configure OS routines
import os

# Configure the plotting routines
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt


from mongo import AnimalShelter
from search import build_sorted_index, binary_search

###########################
# Data Manipulation / Model
###########################

# Connect to database via CRUD Module.
# This connects to my local MongoDB (localhost:27017), which has no username or password.
db = AnimalShelter()

# class read method must support return of list object and accept projection json input
# sending the read method an empty document requests all documents be returned
df = pd.DataFrame.from_records(db.read({}))

# MongoDB v5+ is going to return the '_id' column and that is going to have an 
# invlaid object type of 'ObjectID' - which will cause the data_table to crash - so we remove
# it in the dataframe here. The df.drop command allows us to drop the column. If we do not set
# inplace=True - it will reeturn a new dataframe that does not contain the dropped column(s)
df.drop(columns=['_id'],inplace=True)

## Debug
#print(len(df.to_dict(orient='records')))
#print(df.columns)


#########################
# Dashboard Layout / View
#########################
app = Dash(__name__)

image_filename = 'Grazioso Salvare Logo.png'
encoded_image = base64.b64encode(open(image_filename, 'rb').read())

#FIX ME Also remember to include a unique identifier such as your name or date
#html.Img(src='data:image/png;base64,{}'.format(encoded_image.decode()))

app.layout = html.Div([
#    html.Div(id='hidden-div', style={'display':'none'}),
    html.Center(html.A(html.Img(src='data:image/png;base64,{}'.format(encoded_image.decode())), href="https://www.snhu.edu")),
    html.Center(html.B(html.H3('Lucas Silva - Rescue Dog Search'))),
    html.Hr(),
    html.Div(
        dcc.RadioItems(
            options={
                'WTR': 'Water Rescue',
                'MWR': 'Mountain or Wilderness Rescule',
                'DITR': 'Disaster or Individual Tracking',
                'NONE': 'None',
            },
            value='NONE',
            inline=True,
            id='filter-type',
        ),

    ),
    html.Hr(),
    # Search section: choose Animal ID or Name and type the exact value to find
    html.Div([
        dcc.Dropdown(
            options={
                'animal_id': 'Animal ID',
                'name': 'Name',
            },
            value='animal_id',
            clearable=False,
            id='search-field',
            style={'width': '200px', 'display': 'inline-block'},
        ),
        dcc.Input(
            id='search-value',
            type='text',
            placeholder='Exact Animal ID or Name',
            debounce=True,
            style={'marginLeft': '10px'},
        ),
    ]),
    html.Hr(),
    dash_table.DataTable(id='datatable-id',
                         columns=[{"name": i, "id": i, "deletable": False, "selectable": True} for i in df.columns],
                         data=df.to_dict('records'),
                        # Let you filter by column
                        filter_action="native",
                        # Sort by column
                        sort_action="native",
                        # Pagination
                        page_action="native",
                        page_current=0,
                        page_size=20,
                        row_selectable="single",
                        selected_rows=[0],

                        ),
    html.Br(),
    html.Hr(),
#This sets up the dashboard so that your chart and your geolocation chart are side-by-side
    html.Div(className='row',
         style={'display' : 'flex'},
             children=[
        html.Div(
            id='graph-id',
            className='col s12 m6',

            ),
        html.Div(
            id='map-id',
            className='col s12 m6',
            )
        ])
])

#############################################
# Interaction Between Components / Controller
#############################################



    
# Saves the records for each rescue filter after the first time we load them,
# so we do not have to ask MongoDB again every time the user searches
records_cache = {}
# Saves the sorted lists for each (filter, field) pair. Sorting takes O(n log n),
# which is slower than one linear search, so we only want to sort one time and
# then reuse the sorted list for every search after that.
index_cache = {}

def load_records(filter_type):
    if filter_type == 'WTR':
        records = db.read({
            "animal_type": "Dog",
            "breed": {"$in": [
                "Labrador Retriever Mix",
                "Chesapeake Bay Retriever",
                "Newfoundland"
            ]},
            "sex_upon_outcome": "Intact Female",
            "age_upon_outcome_in_weeks": {"$gte": 26, "$lte": 156}
        })
    elif filter_type == 'MWR':
        records = db.read({
            "animal_type": "Dog",
            "breed": {
                "$in": [
                    "German Shepherd",
                    "Alaskan Malamute",
                    "Old English Sheepdog",
                    "Siberian Husky",
                    "Rottweiler"
                ]
            },
            "sex_upon_outcome": "Intact Male",
            "age_upon_outcome_in_weeks": {
                "$gte": 26,
                "$lte": 156
            }
        })
    elif filter_type == 'DITR':
        records = db.read({
            "animal_type": "Dog",
            "breed": {
                "$in": [
                    "Doberman Pinscher",
                    "German Shepherd",
                    "Golden Retriever",
                    "Bloodhound",
                    "Rottweiler"
                ]
            },
            "sex_upon_outcome": "Intact Male",
            "age_upon_outcome_in_weeks": {
                "$gte": 20,
                "$lte": 300
            }
        })
    else:
        records = db.read({"animal_type": "Dog"})

    # Remove the _id field because the data table cannot show an ObjectId
    for record in records:
        record.pop('_id', None)
    return records

@app.callback(Output('datatable-id','data'),
              [Input('filter-type', 'value'),
               Input('search-field', 'value'),
               Input('search-value', 'value')])
def update_dashboard(filter_type, search_field, search_value):
    if filter_type not in records_cache:
        records_cache[filter_type] = load_records(filter_type)
    records = records_cache[filter_type]

    # If the search box is empty, just show all the records for this filter
    if search_value is None or search_value.strip() == "":
        return records

    key = (filter_type, search_field)
    if key not in index_cache:
        index_cache[key] = build_sorted_index(records, search_field)
    return binary_search(index_cache[key], search_value)

# Display the breeds of animal based on quantity represented in
# the data table
@app.callback(
    Output('graph-id', "children"),
    [Input('datatable-id', "derived_virtual_data")])
def update_graphs(viewData):
    # If the search did not find anything there is nothing to put in the chart
    if not viewData:
        return html.P("No animals match the current search.")
    dff = pd.DataFrame(viewData)
    
    counts = dff.groupby("breed").size().reset_index(name="count")
    
    fig = px.pie(counts, values="count", names="breed", title="Preferred Animals")
    fig.update_traces(textinfo="none")
    
    return dcc.Graph(figure=fig)
    
#This callback will highlight a cell on the data table when the user selects it
@app.callback(
    Output('datatable-id', 'style_data_conditional'),
    [Input('datatable-id', 'selected_columns')]
)
def update_styles(selected_columns):
    return [{
        'if': { 'column_id': i },
        'background_color': '#D2F3FF'
    } for i in selected_columns]


# This callback will update the geo-location chart for the selected data entry
# derived_virtual_data will be the set of data available from the datatable in the form of 
# a dictionary.
# derived_virtual_selected_rows will be the selected row(s) in the table in the form of
# a list. For this application, we are only permitting single row selection so there is only
# one value in the list.
# The iloc method allows for a row, column notation to pull data from the datatable
@app.callback(
    Output('map-id', "children"),
    [Input('datatable-id', "derived_virtual_data"),
     Input('datatable-id', "derived_virtual_selected_rows")])
def update_map(viewData, index):  
    if viewData is None:
        return
    elif index is None:
        return
    # If the search did not find any animals there is nothing to show on the map
    elif len(viewData) == 0:
        return
    
    dff = pd.DataFrame.from_dict(viewData)
    # Because we only allow single row selection, the list can be converted to a row index here
    if index is None:
        row = 0
    else: 
        row = index[0]
    # After a search the table can have fewer rows, so the row that was selected
    # before might not exist anymore. In that case we just use the first row.
    if row >= len(dff):
        row = 0
        
    # Austin TX is at [30.75,-97.48]
    return [
        dl.Map(style={'width': '1000px', 'height': '500px'}, center=[30.75,-97.48], zoom=10, children=[
            dl.TileLayer(id="base-layer-id"),
            # Marker with tool tip and popup
            # Column 13 and 14 define the grid-coordinates for the map
            # Column 4 defines the breed for the animal
            # Column 9 defines the name of the animal
            dl.Marker(position=[dff.iloc[row,13],dff.iloc[row,14]], children=[
                dl.Tooltip(dff.iloc[row,4]),
                dl.Popup([
                    html.H1("Animal Name"),
                    html.P(dff.iloc[row,9])
                ])
            ])
        ])
    ]



app.run(debug=True)


Dash app running on http://127.0.0.1:9771/
